# Year-controlled lyrics, TF-IDF sensitivity and hurdle analysis

**Purpose.** This notebook adds the supervisor-requested entry-year control to the transparent lexical and H3 analyses. The validated TF-IDF work remains a sensitivity and is not retuned.

**Principal inputs.** Clean Y52 cohort; frozen lexical definitions and folds; historical TF-IDF outputs; `supervisor_confirmed_update_v1`.

**Main outputs.** Reconciled LEX_YEAR and H3_YEAR predictions, updated lexical uncertainty, and preserved TF-IDF evidence.


In [ ]:
from pathlib import Path
import json
import math
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

# Find the repository from the current working directory so the notebook
# remains portable when the project folder is moved or opened in Colab.
current = Path.cwd().resolve()
candidates = [current, *current.parents]
ROOT = next((p for p in candidates if (p / "scripts/thesis_utils_v2.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from within the Spotify_clv project.")
sys.path.insert(0, str(ROOT / "scripts"))

from thesis_utils_v2 import (
    SEED, assert_close, assert_unique_tracks, blocked_predictor_hits,
    concise_table, environment_table, extract_spotify_id, load_json,
    read_chart_archive, record_values, regression_metrics,
    save_and_show_figure, setup_notebook,
)

ROOT, FIGURE_DIR, RECORD_DIR = setup_notebook("04_lyrics_tfidf_and_hurdle.ipynb")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
display(environment_table())
print(f"Random seed: {SEED}")


## Lyric quality and related versions

The provisional quality rule requires non-missing, non-blank, non-placeholder lyrics with at least 50 characters and 10 words. It is an audit rule rather than an automatic deletion of unusual songs. All 1,014 primary tracks satisfy it. Exact duplicate lyric text is retained at track level, while protected validation groups prevent supported duplicate/version sets from crossing folds.


In [ ]:
lyric_quality = pd.read_csv(ROOT / "data/processed/pre_model_analysis_ready_v1/lyric_quality_summary.csv")
duplicate_coverage = pd.read_csv(
    ROOT / "data/processed/pre_model_remaining_eda_v2/duplicate_lyric_cohort_coverage_summary.csv"
)
version_summary = pd.read_csv(
    ROOT / "data/processed/final_pre_model_specification_v1/analysis_unit_version_summary.csv"
)
display(lyric_quality)
display(duplicate_coverage)
display(version_summary.head(12))
assert int(lyric_quality.loc[lyric_quality.cohort.eq("550k_52w"), "usable_lyrics"].iloc[0]) == 1014


## Seven transparent lexical summaries

The LEX block adds transformed word count, transformed unique-token count, lexical diversity, average token length, transformed line count, repeated-line proportion and a transformed punctuation-intensity rate. These features describe length, vocabulary and repetition; they do not attempt semantic interpretation.


In [ ]:
import run_provisional_lyrics_hurdle_modelling as lyric_models
import run_supervisor_confirmed_update as supervisor_update

definitions = pd.read_csv(
    ROOT / "data/processed/provisional_lyrics_hurdle_modelling_v1/lyric_feature_definitions.csv"
)
display(definitions)

data, assignments, temporal_assignments = supervisor_update.load_model_data()
assert blocked_predictor_hits(supervisor_update.MODEL_VARIABLES["LEX_YEAR"]) == []
assert "chart_entry_year" in supervisor_update.MODEL_VARIABLES["LEX_YEAR"]
lexical_summary = data[lyric_models.LEX_MODEL].describe(percentiles=[.05, .25, .5, .75, .95]).T
display(lexical_summary[["mean", "std", "min", "5%", "50%", "95%", "max"]].round(3))


Log1p transformations are used for the skewed count/rate variables. They are fixed target-independent definitions; any learned scaling is fitted inside the training fold.

## Recomputed F3_YEAR and LEX_YEAR predictions

The same five protected folds are used for both models, so their errors can be compared track by track. The categorical year encoder and all scaling are fitted separately within each training fold.


In [ ]:
prediction_parts = []
for fold in sorted(data.fold.unique()):
    train = data[data.fold.ne(fold)].copy()
    heldout = data[data.fold.eq(fold)].copy()
    for model_id in ["F3_YEAR", "LEX_YEAR"]:
        prediction, _, fitted_model, warning_text = supervisor_update.fit_count(
            train, heldout, supervisor_update.MODEL_VARIABLES[model_id]
        )
        if warning_text:
            warnings.warn(warning_text)
        prediction_parts.append(pd.DataFrame({
            "spotify_track_id": heldout.spotify_track_id.to_numpy(),
            "fold": fold,
            "primary_group_component": heldout.primary_group_component.to_numpy(),
            "model_id": model_id,
            "y_observed": heldout.Y_52.to_numpy(),
            "y_pred_raw": prediction,
            "X": heldout.X.to_numpy(),
        }))
lex_predictions = pd.concat(prediction_parts, ignore_index=True)
assert not lex_predictions.duplicated(["spotify_track_id", "model_id"]).any()
lex_metrics = pd.DataFrame([
    {"model_id": model_id, **regression_metrics(part.y_observed, part.y_pred_raw)}
    for model_id, part in lex_predictions.groupby("model_id")
])

saved_all = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/year_controlled_grouped_cv_predictions.csv"
)
saved_lex = saved_all[saved_all.model_id.isin(["F3_YEAR", "LEX_YEAR"])]
aligned_lex = lex_predictions.merge(
    saved_lex[["spotify_track_id", "model_id", "fold", "y_observed", "y_pred_raw"]]
        .rename(columns={"y_observed": "saved_outcome", "y_pred_raw": "saved_prediction"}),
    on=["spotify_track_id", "model_id", "fold"], validate="one_to_one"
)
assert len(aligned_lex) == 2 * 1014
assert np.allclose(aligned_lex.y_observed, aligned_lex.saved_outcome, atol=0)
assert np.allclose(aligned_lex.y_pred_raw, aligned_lex.saved_prediction, atol=1e-10)
display(lex_metrics.round(3))

fold_metrics = []
for (fold, model_id), part in lex_predictions.groupby(["fold", "model_id"]):
    fold_metrics.append({"fold": fold, "model_id": model_id,
                         **regression_metrics(part.y_observed, part.y_pred_raw)})
fold_metrics = pd.DataFrame(fold_metrics)
fold_wide = fold_metrics.pivot(index="fold", columns="model_id", values="mae")
fold_wide["F3_YEAR minus LEX_YEAR MAE"] = fold_wide.F3_YEAR - fold_wide.LEX_YEAR
display(fold_wide.round(3))


With entry year held constant, transparent lexical summaries retain a small aggregate MAE advantage over F3_YEAR. The fold table shows whether that increment is consistent, but its practical scale remains modest and RMSE does not establish a comparable advantage.


In [ ]:
uncertainty = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/year_controlled_lexical_uncertainty.csv"
)
display(uncertainty[[
    "metric", "observed_difference", "ci_2_5", "ci_97_5",
    "bootstrap_proportion_candidate_better", "n_tracks", "n_clusters",
]].round(4))

# The X=4 table is retained as historical sensitivity evidence; it was not
# re-estimated because the supervisor requested only the overall year control.
x4_uncertainty = pd.read_csv(
    ROOT / "data/processed/provisional_robustness_uncertainty_v1/x4_paired_uncertainty.csv"
)
x4_lex = x4_uncertainty[
    x4_uncertainty.reference_model.eq("REF_F3")
    & x4_uncertainty.candidate_model.eq("LEX")
]
display(x4_lex[["metric", "observed_difference", "ci_2_5", "ci_97_5", "n_tracks", "n_clusters"]].round(4))


The year-controlled MAE interval now touches zero, while the RMSE interval remains centred near no difference. The bootstrap resamples the same 276 protected components and is conditional on the fixed OOF predictions; it is not a Bayesian probability and does not include model-selection uncertainty. The historical X=4 result remains a qualified sensitivity rather than an updated year-controlled claim.


In [ ]:
tfidf_spec = pd.read_csv(
    ROOT / "data/processed/provisional_lyrics_hurdle_modelling_v1/lyric_model_specifications.csv"
)
tfidf_vocab = pd.read_csv(
    ROOT / "data/processed/provisional_lyrics_hurdle_modelling_v1/tfidf_fold_vocabulary_summary.csv"
)
tfidf_tuning = pd.read_csv(
    ROOT / "data/processed/provisional_lyrics_hurdle_modelling_v1/tfidf_regularisation_summary.csv"
)
display(tfidf_spec[tfidf_spec.model_id.str.startswith("T")])
display(tfidf_vocab[tfidf_vocab.model_id.eq("T3")])
display(tfidf_tuning[tfidf_tuning.model_id.eq("T3")].round(4))

lyric_oof = pd.read_csv(
    ROOT / "data/processed/provisional_lyrics_hurdle_modelling_v1/lyric_grouped_cv_predictions.csv"
)
t3 = lyric_oof[lyric_oof.model_id.eq("T3")]
t3_metrics = regression_metrics(t3.y_observed, t3.y_pred_raw)
display(pd.DataFrame([{"model_id": "T3", **t3_metrics}]).round(3))

lower_alpha = pd.read_csv(
    ROOT / "data/processed/provisional_robustness_uncertainty_v1/tfidf_lower_alpha_sensitivity.csv"
)
display(lower_alpha.round(4))


T3 has the lowest MAE point estimate among the transparent count/text models, but it worsens RMSE and underpredicts on average. Extending alpha below 0.01 did not rescue that pattern. T3 is therefore a high-dimensional lyric sensitivity, not the principal lyric specification.

## H3: future presence and positive longevity

H3 has two components. A logistic model estimates whether `Y_52 > 0`; a zero-truncated Poisson component estimates positive future weeks conditional on presence. Their product forms the combined count prediction. This is a true two-part hurdle implementation, not a zero-inflated model.


In [ ]:
hurdle_parts = []
for fold in sorted(data.fold.unique()):
    train = data[data.fold.ne(fold)].copy()
    heldout = data[data.fold.eq(fold)].copy()
    fitted = supervisor_update.fit_hurdle(
        train, heldout, supervisor_update.MODEL_VARIABLES["H3_YEAR"]
    )
    if fitted["warning_text"]:
        warnings.warn(fitted["warning_text"])
    hurdle_parts.append(pd.DataFrame({
        "spotify_track_id": heldout.spotify_track_id.to_numpy(),
        "fold": fold,
        "model_id": "H3_YEAR",
        "y_observed": heldout.Y_52.to_numpy(),
        "y_pred_raw": fitted["prediction"],
        "p_positive": fitted["probability_positive"],
        "conditional_positive_mean": fitted["conditional_positive_mean"],
    }))
h3 = pd.concat(hurdle_parts, ignore_index=True)
saved_h3 = saved_all[saved_all.model_id.eq("H3_YEAR")]
aligned_h3 = h3.merge(
    saved_h3[["spotify_track_id", "model_id", "fold", "y_observed", "y_pred_raw", "p_positive"]]
        .rename(columns={"y_observed": "saved_outcome", "y_pred_raw": "saved_prediction",
                         "p_positive": "saved_probability"}),
    on=["spotify_track_id", "model_id", "fold"], validate="one_to_one"
)
assert len(aligned_h3) == 1014
assert np.allclose(aligned_h3.y_observed, aligned_h3.saved_outcome, atol=0)
assert np.allclose(aligned_h3.y_pred_raw, aligned_h3.saved_prediction, atol=1e-10)
assert np.allclose(aligned_h3.p_positive, aligned_h3.saved_probability, atol=1e-10)

h3_count_metrics = regression_metrics(h3.y_observed, h3.y_pred_raw)
h3_binary_values = supervisor_update.binary_metrics(
    h3.y_observed.gt(0).astype(int), h3.p_positive, threshold=0.50
)
binary_metrics = pd.DataFrame([{"aggregation": "all_oof", "model_id": "H3_YEAR", **h3_binary_values}])
display(pd.DataFrame([{"model_id": "H3_YEAR combined count", **h3_count_metrics}]).round(3))
display(binary_metrics.round(4))

matrix = np.array([[
    h3_binary_values["true_negative"], h3_binary_values["false_positive"]
], [
    h3_binary_values["false_negative"], h3_binary_values["true_positive"]
]])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Predicted zero", "Predicted positive"],
            yticklabels=["Observed zero", "Observed positive"], ax=ax)
ax.set_title("H3_YEAR grouped OOF confusion matrix (threshold 0.50)")
save_and_show_figure(fig, FIGURE_DIR / "nb04_h3_year_confusion_matrix.png")

h3["calibration_bin"] = pd.qcut(h3.p_positive.rank(method="first"), q=5, labels=False) + 1
cal = h3.groupby("calibration_bin", as_index=False).agg(
    mean_predicted_probability=("p_positive", "mean"),
    observed_positive_rate=("y_observed", lambda value: value.gt(0).mean()),
    tracks=("spotify_track_id", "size"),
)
display(cal.round(3))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(cal.mean_predicted_probability, cal.observed_positive_rate,
        marker="o", color="#4C78A8")
ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
ax.set(xlabel="Mean predicted P(Y52 > 0)", ylabel="Observed positive proportion",
       title="H3_YEAR future-presence calibration")
save_and_show_figure(fig, FIGURE_DIR / "nb04_h3_year_probability_calibration.png")


H3_YEAR retains strong discrimination and useful probability calibration for any future presence. Its expected-count error remains close to LEX_YEAR rather than clearly better. The positive-duration component is still the more difficult part, so the distinction between continuation and exact longevity remains central.

## Notebook 04 conclusions

Entry-year control does not promote high-dimensional TF-IDF or change the role of H3. The principal lyric claim remains limited to a small, interpretable lexical increment; TF-IDF stays in sensitivity material and H3 stays a companion probability/two-part analysis.


In [ ]:
lex_index = lex_metrics.set_index("model_id")
h3_binary = binary_metrics.iloc[0]
lex_mae = uncertainty[uncertainty.metric.eq("MAE")].iloc[0]
lex_rmse = uncertainty[uncertainty.metric.eq("RMSE")].iloc[0]
record_values(RECORD_DIR / "nb04_values.json", "04_lyrics_tfidf_and_hurdle", {
    "lex_year_mae": lex_index.loc["LEX_YEAR", "mae"],
    "lex_year_rmse": lex_index.loc["LEX_YEAR", "rmse"],
    "f3_year_mae": lex_index.loc["F3_YEAR", "mae"],
    "f3_year_rmse": lex_index.loc["F3_YEAR", "rmse"],
    "t3_mae_historical": t3_metrics["mae"],
    "t3_rmse_historical": t3_metrics["rmse"],
    "h3_year_mae": h3_count_metrics["mae"],
    "h3_year_rmse": h3_count_metrics["rmse"],
    "h3_year_accuracy": h3_binary.accuracy,
    "h3_year_precision": h3_binary.precision_positive_predictive_value,
    "h3_year_recall": h3_binary.recall_sensitivity,
    "h3_year_specificity": h3_binary.specificity,
    "h3_year_f1_score": h3_binary.classification_f1_score,
    "h3_year_roc_auc": h3_binary.roc_auc,
    "h3_year_pr_auc": h3_binary.pr_auc,
    "f3_year_lex_year_mae_difference": lex_mae.observed_difference,
    "f3_year_lex_year_mae_ci_low": lex_mae.ci_2_5,
    "f3_year_lex_year_mae_ci_high": lex_mae.ci_97_5,
    "f3_year_lex_year_rmse_difference": lex_rmse.observed_difference,
    "f3_year_lex_year_rmse_ci_low": lex_rmse.ci_2_5,
    "f3_year_lex_year_rmse_ci_high": lex_rmse.ci_97_5,
    "maximum_saved_lex_prediction_difference": float((aligned_lex.y_pred_raw - aligned_lex.saved_prediction).abs().max()),
    "maximum_saved_h3_prediction_difference": float((aligned_h3.y_pred_raw - aligned_h3.saved_prediction).abs().max()),
})
print("Notebook 04 year-controlled reconciliation record saved.")
